In [1]:
import numpy as np
import scanpy as sc
import pandas as pd
import os
from matplotlib import pyplot as plt
from umap import UMAP
import seaborn as sns
import sys; sys.path += ['./../../']
from CardamomOT import NetworkModel as CardamomNetworkModel
from CardamomOT import train_classifier, predict_cell_types
from harissa import NetworkModel as HarissaNetworkModel

In [2]:
def run_inference(rna_data, time, deg_rates):
    # ---- CardamomOT fit ----
    x = rna_data[1:, 1:].copy()
    x[:, 0] = time
    G = x.shape[1]
    model = CardamomNetworkModel(G - 1)
    model.d = deg_rates
    model.fit(x)
    # model.adapt_to_unitary()

    return model

# FN4

In [49]:
np.random.seed(0)

# Number of cells
C = 1000
t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96]
N = int(C /len(t))
k = np.linspace(0, C, len(t) + 1, dtype='int')
print(f't = {t}')
time = np.zeros(C, dtype='int')
for i in range(len(t)): time[k[i]:k[i+1]] = t[i]

# Number of genes
G = 4

# Prepare data
data = np.zeros((C+1,G+2), dtype='int')
data[0, 1:] = np.arange(G+1)
data[1:,0] = time # Time points
data[1:,1] = 100 * (time > 0) # Stimulus

# Initialize the model
model_harissa_FN4 = HarissaNetworkModel(G)
model_harissa_FN4.d[0] = 1
model_harissa_FN4.d[1] = 0.2
model_harissa_FN4.d /= 5

model_harissa_FN4.basal[1:] = -5
model_harissa_FN4.inter[0,1] = 10
model_harissa_FN4.inter[1,2] = 10
model_harissa_FN4.inter[1,3] = 10
model_harissa_FN4.inter[3,4] = 10
model_harissa_FN4.inter[4,1] = -10
model_harissa_FN4.inter[2,2] = 10
model_harissa_FN4.inter[3,3] = 10

inter_true = 1 * (abs(model_harissa_FN4.inter) > 0)

prot_traj_FN4 = np.ones((C, G+1), dtype='float32')
prot_traj_FN4[:, 0] = (time > 0).astype('float32')
for k in range(C):
    sim = model_harissa_FN4.simulate(time[k], burnin=5)
    prot_traj_FN4[k,1:] = sim.p[-1]
    data[k+1,2:] = np.random.poisson(sim.m[-1])
rna_traj_FN4 = data[1:, 1:].copy()

model_FN4 = run_inference(data, time, model_harissa_FN4.d.copy())

t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96]
Calibrating gene 2
Calibrating gene 1
Calibrating gene 3
Calibrating gene 4
Gene 1-1 calibrated... [0.112 1.566] 0.0172190631887181
Gene 2-2 calibrated... [0.018 1.753] 0.018251007133238156
Gene 3-3 calibrated... [0.014 1.78 ] 0.018672353431554864
Gene 4-4 calibrated... [0.012 1.336] 0.01804282075862895
Mean proba =  0.9702512965853897 1.0
[fit_network] Cell counts per sample/timepoint and genes:
 [[100 100 100 100 100 100 100 100 100 100]] 5
[fit_network] Number of simulated cells per sample: [100]
[fit_network] Number of total cells per sample: [100]
1 0 | Errors (before, after): 0.68194, 0.22041 | alpha mean: 0.0100
number of non reached cells 337
2 0 | Errors (before, after): 0.19852, 0.19118 | alpha mean: 0.0388
number of non reached cells 336
3 0 | Errors (before, after): 0.18155, 0.17849 | alpha mean: 0.0523
number of non reached cells 328
4 0 | Errors (before, after): 0.16777, 0.16517 | alpha mean: 0.0579
number of non reached cells 31

# CN5

In [4]:
np.random.seed(0)

# Number of cells
C = 1000
t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96]
N = int(C /len(t))
k = np.linspace(0, C, len(t) + 1, dtype='int')
print(f't = {t}')
time = np.zeros(C, dtype='int')
for i in range(len(t)): time[k[i]:k[i+1]] = t[i]

# Number of genes
G = 5

# Prepare data
data = np.zeros((C+1,G+2), dtype='int')
data[0][1:] = np.arange(G+1)
data[1:,0] = time # Time points
data[1:,1] = 100 * (time > 0) # Stimulus

# Initialize the model
model_harissa_CN5 = HarissaNetworkModel(G)
model_harissa_CN5.d[0] = 0.5
model_harissa_CN5.d[1] = 0.1

model_harissa_CN5.basal[1:] = [-5, 4, 4, -5, -5]
model_harissa_CN5.inter[0, 1] = 10
model_harissa_CN5.inter[1, 2] = -10
model_harissa_CN5.inter[2, 3] = -10
model_harissa_CN5.inter[3, 4] = 10
model_harissa_CN5.inter[4, 5] = 10
model_harissa_CN5.inter[5, 1] = -10

inter_true = 1 * (abs(model_harissa_CN5.inter) > 0)

prot_traj_CN5 = np.ones((C, G+1), dtype='float32')
prot_traj_CN5[:, 0] = (time > 0).astype('float32')
for k in range(C):
    sim = model_harissa_CN5.simulate(time[k], burnin=5)
    prot_traj_CN5[k,1:] = sim.p[-1]
    data[k+1,2:] = np.random.poisson(sim.m[-1])
rna_traj_CN5 = data[1:,1:].copy()

model_CN5 = run_inference(data, time, model_harissa_CN5.d.copy())

t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96]
Calibrating gene 1
Calibrating gene 2
Calibrating gene 3
Calibrating gene 4
Calibrating gene 5
Gene 1-1 calibrated... [0.13  1.827] 0.019451378775687596
Gene 2-2 calibrated... [0.12  1.504] 0.018638369742186123
Gene 3-3 calibrated... [0.244 1.801] 0.019696314689503593
Gene 4-4 calibrated... [0.096 1.338] 0.016198408329014327
Gene 5-5 calibrated... [0.068 1.819] 0.020339950994837697
Mean proba =  0.9073021354979824 1.0
[fit_network] Cell counts per sample/timepoint and genes:
 [[100 100 100 100 100 100 100 100 100 100]] 6
[fit_network] Number of simulated cells per sample: [100]
[fit_network] Number of total cells per sample: [100]
1 0 | Errors (before, after): 0.67398, 0.39841 | alpha mean: 0.0100
number of non reached cells 350
2 0 | Errors (before, after): 0.39328, 0.38923 | alpha mean: 0.1766
number of non reached cells 331
3 0 | Errors (before, after): 0.36810, 0.34982 | alpha mean: 0.3051
number of non reached cells 325
4 0 | Errors (befo

# BN8

In [5]:
np.random.seed(0)

# Number of cells
C = 1000

# Time points
t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96] 
N = int(C /len(t))
k = np.linspace(0, C, len(t) + 1, dtype='int')
print(f't = {t}')
time = np.zeros(C, dtype='int')
for i in range(len(t)): time[k[i]:k[i+1]] = t[i]

# Number of genes
G = 8

# Prepare data
data = np.zeros((C+1,G+2), dtype='int')
data[0][1:] = np.arange(G+1)
data[1:,0] = time # Time points
data[1:,1] = 100 * (time > 0) # Stimulus

# Initialize the model
model_harissa_BN8 = HarissaNetworkModel(G)
model_harissa_BN8.d[0] = 0.25
model_harissa_BN8.d[1] = 0.05

model_harissa_BN8.basal[1:] = [-4, -4, -4, -4, -4, -4, -4, -4]
model_harissa_BN8.inter[0, 1] = 10
model_harissa_BN8.inter[1, 2] = 10
model_harissa_BN8.inter[1, 3] = 10
model_harissa_BN8.inter[3, 2] = -10
model_harissa_BN8.inter[2, 3] = -10
model_harissa_BN8.inter[2, 2] = 5
model_harissa_BN8.inter[3, 3] = 5
model_harissa_BN8.inter[2, 4] = 10
model_harissa_BN8.inter[3, 5] = 10
model_harissa_BN8.inter[2, 5] = -10
model_harissa_BN8.inter[3, 4] = -10
model_harissa_BN8.inter[4, 7] = -10
model_harissa_BN8.inter[5, 6] = -10
model_harissa_BN8.inter[4, 6] = 10
model_harissa_BN8.inter[5, 7] = 10
model_harissa_BN8.inter[7, 8] = 10
model_harissa_BN8.inter[6, 8] = -10

inter_true = 1 * (abs(model_harissa_BN8.inter) > 0)

prot_traj_BN8 = np.ones((C, G+1), dtype='float32')
prot_traj_BN8[:, 0] = (time > 0).astype('float32')
for k in range(C):
    sim = model_harissa_BN8.simulate(time[k], burnin=5)
    prot_traj_BN8[k,1:] = sim.p[-1]
    data[k+1,2:] = np.random.poisson(sim.m[-1])
rna_traj_BN8 = data[1:,1:].copy()

model_BN8 = run_inference(data, time, model_harissa_BN8.d.copy())

t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96]
Calibrating gene 1
Calibrating gene 2
Calibrating gene 3
Calibrating gene 4
Calibrating gene 5
Calibrating gene 6
Calibrating gene 7
Calibrating gene 8
Gene 1-1 calibrated... [0.009 1.775] 0.017535021234671863
Gene 2-2 calibrated... [0.076 1.74 ] 0.018075420185728577
Gene 3-3 calibrated... [0.09  1.725] 0.018944214983840777
Gene 4-4 calibrated... [0.035 1.726] 0.018090608408309138
Gene 5-5 calibrated... [0.023 1.729] 0.021365358796889382
Gene 6-6 calibrated... [0.016 0.915] 0.014225639702475819
Gene 7-7 calibrated... [0.022 1.112] 0.014891113608224581
Gene 8-8 calibrated... [0.006 0.476] 0.01
Mean proba =  0.9557609930620774 1.0
[fit_network] Cell counts per sample/timepoint and genes:
 [[100 100 100 100 100 100 100 100 100 100]] 9
[fit_network] Number of simulated cells per sample: [100]
[fit_network] Number of total cells per sample: [100]
1 0 | Errors (before, after): 0.69732, 0.34959 | alpha mean: 0.0100
number of non reached cells 349
2 0

# FN8

In [6]:
np.random.seed(0)

# Number of cells
C = 1000
# Time points
t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96] # np.linspace(0, 25, 10, dtype='int')
N = int(C /len(t))
k = np.linspace(0, C, len(t) + 1, dtype='int')
print(f't = {t}')
time = np.zeros(C, dtype='int')
for i in range(len(t)): time[k[i]:k[i+1]] = t[i]

# Number of genes
G = 8

# Prepare data
data = np.zeros((C+1,G+2), dtype='int')
data[0][1:] = np.arange(G+1)
data[1:,0] = time # Time points
data[1:,1] = 100 * (time > 0) # Stimulus

# Initialize the model
model_harissa_FN8 = HarissaNetworkModel(G)
model_harissa_FN8.d[0] = 0.4
model_harissa_FN8.d[1] = 0.08

model_harissa_FN8.basal[1:] = [-5, -5, -5, -5, -5, -5, -5, -5]
model_harissa_FN8.inter[0, 1] = 10
model_harissa_FN8.inter[1, 2] = 10
model_harissa_FN8.inter[2, 3] = 10
model_harissa_FN8.inter[3, 4] = 10
model_harissa_FN8.inter[3, 5] = 10
model_harissa_FN8.inter[3, 6] = 10
model_harissa_FN8.inter[4, 1] = -10
model_harissa_FN8.inter[5, 1] = -10
model_harissa_FN8.inter[6, 1] = -10
model_harissa_FN8.inter[4, 4] = 10
model_harissa_FN8.inter[5, 5] = 10
model_harissa_FN8.inter[6, 6] = 10
model_harissa_FN8.inter[4, 8] = -10
model_harissa_FN8.inter[4, 7] = -10
model_harissa_FN8.inter[6, 7] = 10
model_harissa_FN8.inter[7, 6] = 10
model_harissa_FN8.inter[8, 8] = 10

inter_true = 1 * (abs(model_harissa_FN8.inter) > 0)

prot_traj_FN8 = np.ones((C, G+1), dtype='float32')
prot_traj_FN8[:, 0] = (time > 0).astype('float32')
for k in range(C):
    sim = model_harissa_FN8.simulate(time[k], burnin=5)
    prot_traj_FN8[k,1:] = sim.p[-1]
    data[k+1,2:] = np.random.poisson(sim.m[-1])
rna_traj_FN8 = data[1:,1:].copy()

model_FN8 = run_inference(data, time, model_harissa_FN8.d.copy())

t = [0, 6, 12, 24, 36, 48, 60, 72, 84, 96]
Calibrating gene 1
Calibrating gene 2
Calibrating gene 3
Calibrating gene 4
Calibrating gene 5
Calibrating gene 6
Calibrating gene 7
Calibrating gene 8
Gene 1-1 calibrated... [0.046 1.703] 0.01813634629696305
Gene 2-2 calibrated... [0.058 1.38 ] 0.017423013853610503
Gene 3-3 calibrated... [0.053 1.425] 0.018915772139665527
Gene 4-4 calibrated... [0.02  1.652] 0.016347718542492017
Gene 5-5 calibrated... [0.025 2.029] 0.02090623124082012
Gene 6-6 calibrated... [0.026 2.003] 0.021485709385067542
Gene 7-7 calibrated... [0.008 0.496] 0.01211493347362812
Gene 8-8 calibrated... [0.007 0.43 ] 0.02997221655926822
Mean proba =  0.9598835343747214 1.0
[fit_network] Cell counts per sample/timepoint and genes:
 [[100 100 100 100 100 100 100 100 100 100]] 9
[fit_network] Number of simulated cells per sample: [100]
[fit_network] Number of total cells per sample: [100]
1 0 | Errors (before, after): 0.69791, 0.25212 | alpha mean: 0.0100
number of non reached c

# Table

In [78]:
def kon(model_harissa, p):
    from scipy.special import expit
    """
    Interaction function kon (off->on rate), given protein levels p.
    """
    sigma = expit(model_harissa.basal + p @ model_harissa.inter)
    Kon = sigma
    Kon[0] = 0 # Ignore stimulus
    return Kon

def correctly_attributed(model_harissa, prot_traj_harissa, model_cardamom):
    kon_harissa = kon(model_harissa, prot_traj_harissa)[:,1:]
    modes_cardamom = model_cardamom.modes[:,1:] #Ignore stimulus
    kon_bin = (kon_harissa > 0.5).astype('int')
    modes_bin = (modes_cardamom > 0.5).astype('int')
    return (kon_bin == modes_bin).mean()

In [100]:
def _format_ref(values: np.ndarray) -> str:
    mean = float(np.mean(values))
    return f"{mean:.3f}".rstrip("0").rstrip(".")


def _format_inf(values: np.ndarray) -> str:
    mean = float(np.mean(values))
    std = float(np.std(values))
    return f"{mean:.3f}$\\pm${std:.3f}"


def _ratio_values(net_model, a_idx: int) -> np.ndarray:
    a = np.asarray(net_model.a)
    d = np.asarray(net_model.d)

    if a.ndim == 1:
        num = np.atleast_1d(a[a_idx]).astype(float)
    else:
        num = np.asarray(a[a_idx, 1:], dtype=float)

    if d.ndim == 1:
        den = np.full(num.shape, float(d[0]))
    else:
        den = np.asarray(d[0, 1:], dtype=float)

    return np.asarray(num / den, dtype=float).ravel()


def _a_off_values(net_model) -> np.ndarray:
    a = np.asarray(net_model.a)

    if a.ndim == 1:
        values = np.atleast_1d(a[-1]).astype(float)
    else:
        values = np.asarray(a[-1, 1:], dtype=float)

    return np.asarray(values, dtype=float).ravel()


networks = {
    "FN4": (model_harissa_FN4, model_FN4),
    "CN5": (model_harissa_CN5, model_CN5),
    "BN8": (model_harissa_BN8, model_BN8),
    "FN8": (model_harissa_FN8, model_FN8),
}

prot_trajectories = {
    "FN4": prot_traj_FN4,
    "CN5": prot_traj_CN5,
    "BN8": prot_traj_BN8,
    "FN8": prot_traj_FN8,
}

k0_ref_inf = {}
k1_ref_inf = {}
koff_ref_inf = {}
attrib_inf = {}
for name, (model_harissa, model) in networks.items():
    k0_ref_values = _ratio_values(model_harissa, 0)
    k1_ref_values = _ratio_values(model_harissa, 1)

    k0_inf_values = _ratio_values(model, 0)
    k1_inf_values = _ratio_values(model, 1)

    koff_ref_values = _a_off_values(model_harissa)
    koff_inf_values = _a_off_values(model)

    k0_ref_inf[name] = (_format_ref(k0_ref_values), _format_inf(k0_inf_values))
    k1_ref_inf[name] = (_format_ref(k1_ref_values), _format_inf(k1_inf_values))
    koff_ref_inf[name] = (_format_ref(koff_ref_values), _format_inf(koff_inf_values))

    attrib = correctly_attributed(model_harissa, prot_trajectories[name], model)
    attrib_inf[name] = f"{float(attrib):.3f}"

line_break = r"\\"

header = [
    r"\begin{table}[ht]",
    r"\centering",
    r"\caption{Comparison of parameters for different models}",
    r"\small",
    r"\begin{tabular}{|l|cc|cc|cc|cc|}",
    r"\hline",
    r"\multirow{2}{*}{Parameter} & \multicolumn{2}{c|}{FN4} & \multicolumn{2}{c|}{CN5} & \multicolumn{2}{c|}{BN8} & \multicolumn{2}{c|}{FN8} " + line_break,
    r"\cline{2-9}",
    r"& Ref & Inf & Ref & Inf & Ref & Inf & Ref & Inf " + line_break,
    r"\hline",
]

row_k0 = (
    "$k_0 /d_0$"
    f" & {k0_ref_inf['FN4'][0]} & {k0_ref_inf['FN4'][1]}"
    f" & {k0_ref_inf['CN5'][0]} & {k0_ref_inf['CN5'][1]}"
    f" & {k0_ref_inf['BN8'][0]} & {k0_ref_inf['BN8'][1]}"
    f" & {k0_ref_inf['FN8'][0]} & {k0_ref_inf['FN8'][1]} " + line_break
)

row_k1 = (
    "$k_1 /d_0$"
    f" & {k1_ref_inf['FN4'][0]} & {k1_ref_inf['FN4'][1]}"
    f" & {k1_ref_inf['CN5'][0]} & {k1_ref_inf['CN5'][1]}"
    f" & {k1_ref_inf['BN8'][0]} & {k1_ref_inf['BN8'][1]}"
    f" & {k1_ref_inf['FN8'][0]} & {k1_ref_inf['FN8'][1]} " + line_break
)

row_koff = (
    "$k_{off}/s_0$"
    f" & {koff_ref_inf['FN4'][0]} & {koff_ref_inf['FN4'][1]}"
    f" & {koff_ref_inf['CN5'][0]} & {koff_ref_inf['CN5'][1]}"
    f" & {koff_ref_inf['BN8'][0]} & {koff_ref_inf['BN8'][1]}"
    f" & {koff_ref_inf['FN8'][0]} & {koff_ref_inf['FN8'][1]} " + line_break
)

row_attrib = (
    r"\shortstack[l]{\% of correctly\\attributed modes}"
    f" &  & {attrib_inf['FN4']}"
    f" &  & {attrib_inf['CN5']}"
    f" &  & {attrib_inf['BN8']}"
    f" &  & {attrib_inf['FN8']} " + line_break
)

footer = [
    row_koff,
    row_attrib,
    r"\hline",
    r"\end{tabular}",
    r"\label{tableS2}",
    r"\end{table}",
]

newline = chr(10)
latex_table = newline.join(header + [row_k0, row_k1] + footer)
print(latex_table)

\begin{table}[ht]
\centering
\caption{Comparison of parameters for different models}
\small
\begin{tabular}{|l|cc|cc|cc|cc|}
\hline
\multirow{2}{*}{Parameter} & \multicolumn{2}{c|}{FN4} & \multicolumn{2}{c|}{CN5} & \multicolumn{2}{c|}{BN8} & \multicolumn{2}{c|}{FN8} \\
\cline{2-9}
& Ref & Inf & Ref & Inf & Ref & Inf & Ref & Inf \\
\hline
$k_0 /d_0$ & 0 & 0.194$\pm$0.211 & 0 & 0.263$\pm$0.120 & 0 & 0.138$\pm$0.117 & 0 & 0.076$\pm$0.046 \\
$k_1 /d_0$ & 10 & 8.044$\pm$0.888 & 4 & 3.315$\pm$0.401 & 8 & 5.599$\pm$1.870 & 5 & 3.474$\pm$1.445 \\
$k_{off}/s_0$ & 0.02 & 0.018$\pm$0.001 & 0.02 & 0.019$\pm$0.001 & 0.02 & 0.017$\pm$0.003 & 0.02 & 0.019$\pm$0.005 \\
\shortstack[l]{\% of correctly\\attributed modes} &  & 0.929 &  & 0.852 &  & 0.913 &  & 0.926 \\
\hline
\end{tabular}
\label{tableS2}
\end{table}
